In [3]:
import ee
import time
import os
from google.colab import drive

# This will prompt you to log in and give Colab access to your Drive
drive.mount('/content/drive')

# Initialize the Earth Engine library
ee.Authenticate()
ee.Initialize(project='integrated-hawk-485001-k3') # Replace 'YOUR_PROJECT_ID' with your actual Google Cloud Project ID

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# This points directly to your cloud Drive
drive_folder = '/content/drive/MyDrive/Sentinel2_Training_Data'

if not os.path.exists(drive_folder):
    os.makedirs(drive_folder)

In [5]:
# 1. LOAD DHS CLUSTERS
ASSET_ID = 'projects/integrated-hawk-485001-k3/assets/PH_DHS_GPS'
dhs_points = ee.FeatureCollection(ASSET_ID)

In [6]:
# 2. DEFINE ADAPTIVE BUFFER FUNCTION
def adaptive_buffer(feature):
    urban_rural_status = ee.String(feature.get('URBAN_RURA'))
    is_urban = urban_rural_status.compareTo('U').eq(0)
    # 2000m for Urban, 5000m for Rural
    radius = ee.Number(ee.Algorithms.If(is_urban, 2000, 5000))
    return feature.buffer(radius).bounds()

dhs_squares = dhs_points.map(adaptive_buffer)

In [7]:
# 3. DEFINE CLOUD MASKING
def mask_s2_clouds(image):
    qa = image.select('QA60')
    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(mask).divide(10000)

In [8]:
# 4. DEFINE QUARTERS
quarters = {
    1: ('2022-01-01', '2022-03-31'),
    2: ('2022-04-01', '2022-06-30'),
    3: ('2022-07-01', '2022-09-30'),
    4: ('2022-10-01', '2022-12-31')
}

In [9]:
 # ==========================================
# 3. EXPORT LOOP WITH MISSING FILE CHECK
# ==========================================
features_list = dhs_squares.getInfo()['features']
total_tasks = 0
skipped_tasks = 0
task_limit = 2800 # Safety buffer below the 3,000 limit

print(f"Found {len(features_list)} clusters. Scanning Drive for missing files...")

for feature in features_list:
    # Stop if we hit the Earth Engine safety limit
    if total_tasks >= task_limit:
        print(f"Reached safe task limit of {task_limit}. Stopping queue.")
        break

    cluster_id = str(feature['properties']['DHSCLUST'])
    roi_geometry = ee.Geometry.Polygon(feature['geometry']['coordinates'])

    for q_num, (start_date, end_date) in quarters.items():

        task_desc = f"dhs_{cluster_id}_2022_Q{q_num}"
        expected_filepath = os.path.join(drive_folder, f"{task_desc}.tif")

        # SKIP LOGIC: Check Colab's connection to your Drive
        if os.path.exists(expected_filepath):
            skipped_tasks += 1
            continue

        # Build task if missing
        quarterly_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(roi_geometry)
            .filterDate(start_date, end_date)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80)))

        best_layer = (quarterly_col
                      .map(mask_s2_clouds)
                      .select(['B4', 'B3', 'B2', 'B8', 'B11'])
                      .median())

        backup_layer = (quarterly_col
                        .select(['B4', 'B3', 'B2', 'B8', 'B11'])
                        .mosaic()
                        .divide(10000))

        final_img = best_layer.unmask(backup_layer).clip(roi_geometry)

        export_task = ee.batch.Export.image.toDrive(
            image=final_img,
            description=task_desc,
            folder='Sentinel2_Training_Data',
            region=roi_geometry,
            scale=10,
            crs='EPSG:3857',
            fileFormat='GeoTIFF'
        )

        export_task.start()
        total_tasks += 1

print(f"Done scanning! Skipped {skipped_tasks} existing images.")
print(f"Successfully queued {total_tasks} missing tasks in Earth Engine.")

Found 1247 clusters. Scanning Drive for missing files...
Done scanning! Skipped 2736 existing images.
Successfully queued 2252 missing tasks in Earth Engine.
